Reusable H.264 Video Processing Script
This notebook provides a streamlined script to concatenate video_000.h264 (as the reference for header and trimming) with another H.264 video, convert the result to MP4, and then trim the video_000 portion from the beginning.



## Setup: Extract Header and Determine Duration of `video_000.h264`

This initial step is performed once to prepare the necessary information from `video_000.h264`. We extract a generic H.264 header segment and determine its exact duration. These values are then used by the `process_video` function.

In [ ]:
import os
import re

# --- Global Setup (run once) ---
video0_h264_filename = 'video_000.h264'
header_file = 'extracted_header.h264'
temp_video0_mp4 = 'video_000_temp.mp4'
video_000_duration_seconds = None

print(f"--- Initializing for '{video0_h264_filename}' ---")

# 1. Extract Header from video_000.h264
with open(video0_h264_filename, 'rb') as f:
    data = f.read()

start_code = b'\x00\x00\x00\x01'
positions = []
i = 0
while True:
    i = data.find(start_code, i)
    if i == -1:
        break
    positions.append(i)
    i += 4

header_end = positions[5] if len(positions) > 5 else 2000 # Heuristic for header end

with open(header_file, 'wb') as f:
    f.write(data[:header_end])
print(f"Extracted header saved to '{header_file}' (approx. {header_end} bytes).")

# 2. Determine Duration of video_000.h264
print(f'Converting {video0_h264_filename} to {temp_video0_mp4} to extract duration...')
!ffmpeg -y -f h264 -fflags +genpts -r 15 -i {video0_h264_filename} -c:v copy -movflags +faststart {temp_video0_mp4}

print(f'Extracting duration from {temp_video0_mp4}...')
output = !ffmpeg -i {temp_video0_mp4} -f null - 2>&1 | grep Duration
duration_line = output[0]

duration_match = re.search(r'Duration: (\d{2}):(\d{2}):(\d{2}\.\d{2})', duration_line)
if duration_match:
    hours = float(duration_match.group(1))
    minutes = float(duration_match.group(2))
    seconds = float(duration_match.group(3))
    video_000_duration_seconds = hours * 3600 + minutes * 60 + seconds
    print(f'Duration of {video0_h264_filename}: {video_000_duration_seconds} seconds')
else:
    print(f'Could not extract duration of {video0_h264_filename}.')

# Clean up the temporary MP4 file
if os.path.exists(temp_video0_mp4):
    os.remove(temp_video0_mp4)
    print(f"Removed temporary file: {temp_video0_mp4}")

print("--- Setup complete ---")

## `process_video` Function Definition

This function encapsulates the logic for concatenating `video_000.h264` with another video, converting to MP4, and then trimming the `video_000` portion. You can call this function for any secondary H.264 video file.

In [ ]:
def process_video(second_video_h264_filename, output_prefix='processed'):
    """
    Concatenates video_000.h264 with a second H.264 video,
    converts the result to MP4, and then trims the video_000.h264 portion.

    Args:
        second_video_h264_filename (str): The filename of the second H.264 video.
        output_prefix (str): Prefix for the output filenames.

    Returns:
        str: The filename of the final trimmed MP4 video, or None if an error occurred.
    """
    if video_000_duration_seconds is None:
        print("Error: video_000.h264 duration not determined. Please run setup cell first.")
        return None

    concatenated_h264 = f'{output_prefix}_{second_video_h264_filename.replace(".h264", "")}_concat.h264'
    merged_mp4 = f'{output_prefix}_{second_video_h264_filename.replace(".h264", "")}_merged.mp4'
    final_trimmed_mp4 = f'{output_prefix}_{second_video_h264_filename.replace(".h264", "")}_trimmed.mp4'

    print(f"\nProcessing '{second_video_h264_filename}'...")

    # Step 1: Concatenate video_000.h264 and the second video
    print(f"Concatenating '{video0_h264_filename}' and '{second_video_h264_filename}'...")
    # Assuming video0_h264_filename already contains the necessary header.
    # We prepend the extracted header explicitly to ensure consistent headers across all concatenations.
    !cat {header_file} {video0_h264_filename} {second_video_h264_filename} > {concatenated_h264}
    print(f"Concatenated H.264 stream saved to '{concatenated_h264}'.")

    # Step 2: Convert Concatenated H.264 to MP4
    print(f"Converting '{concatenated_h264}' to MP4...")
    !ffmpeg -y -f h264 -fflags +genpts -r 15 -probesize 100M -analyzeduration 100M -i {concatenated_h264} -c:v copy -movflags +faststart {merged_mp4}
    print(f"Converted to '{merged_mp4}'.")

    # Step 3: Trim the Merged MP4
    print(f"Trimming '{merged_mp4}' by {video_000_duration_seconds} seconds (to remove '{video0_h264_filename}' content)...")
    !ffmpeg -y -i {merged_mp4} -ss {video_000_duration_seconds} -c:v copy -c:a copy {final_trimmed_mp4}
    print(f"Trimming complete. Final video saved as '{final_trimmed_mp4}'.")

    # Optional cleanup of intermediate files
    if os.path.exists(concatenated_h264): os.remove(concatenated_h264)
    if os.path.exists(merged_mp4): os.remove(merged_mp4)

    return final_trimmed_mp4

Exam Usage is provided :

In [ ]:
from google.colab import files

# Example for video_008.h264
final_video = process_video('video_008.h264', output_prefix='my_project')

if final_video and os.path.exists(final_video):
    print(f"\nSuccessfully processed: '{final_video}'. Downloading...")
    files.download(final_video)
else:
    print("Processing failed or final video not found.")

# You can repeat this cell or modify the `second_video_h264_filename`
# to process other videos, e.g., process_video('video_009.h264')